# 03c | JOIN

The SQL counterpart of pandas' `merge`. You already learned the concepts in 02d; this page is mostly new syntax, plus two pitfalls specific to SQL.

**After this page you can:** join several tables, keep unmatched rows, find "records with no ...", and write self-joins. About 50 minutes.

## 1. INNER JOIN

```sql
SELECT o.order_id, u.name, o.amount
FROM orders AS o
JOIN users AS u
  ON o.user_id = u.user_id;
```

- `o` and `u` are table aliases (`AS` is optional) that say which table each column comes from;
- The join condition follows `ON`;
- A bare `JOIN` is an `INNER JOIN`: it keeps only rows that match on both sides.

In [ ]:
SELECT o.order_id, u.name, u.country, o.amount
FROM orders o
JOIN users u
  ON o.user_id = u.user_id
LIMIT 5;

### ✏️ Exercise 1 (fill in the blanks)

List each order's `order_id`, the `name` of the user who placed it, and `amount`.

In [ ]:
SELECT o.order_id, u.____, o.amount
FROM orders o
JOIN ____ u
  ON o.user_id = u.____;

<details>
<summary><b>Answer</b></summary>

```sql
SELECT o.order_id, u.name, o.amount
FROM orders o
JOIN users u
  ON o.user_id = u.user_id;
```

</details>

## 2. LEFT JOIN

`LEFT JOIN` keeps **every row of the left table**. Where the right table has no match, its columns are NULL.

`users` has 10 rows and `orders` has 16. The inner join gives 16 rows (one per order); the left join gives 18: the 16 orders plus one row for each of the 2 users who never ordered.

In [ ]:
SELECT u.user_id, u.name, o.order_id, o.amount
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
WHERE u.user_id IN (6, 7, 8);

Just like in pandas: **a join increases the row count because the join key is not unique in the other table.**

## 3. LEFT JOIN + aggregation

"Total spend per user, showing 0 for users with no orders":

- `SUM` over a group that is all NULL returns NULL; use `COALESCE(x, 0)` to turn NULL into 0;
- Count orders with `COUNT(o.order_id)`, not `COUNT(*)`: users with no orders still occupy one row, so `COUNT(*)` would count them as 1.

In [ ]:
SELECT u.user_id, u.name,
       COUNT(*)                   AS wrong_n,
       COUNT(o.order_id)          AS n_orders,
       COALESCE(SUM(o.amount), 0) AS total
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
GROUP BY u.user_id, u.name;

## 4. Pitfall: filters on the right table belong in ON

Now suppose we only want to count **completed** orders. The instinct is to add `WHERE o.status = 'completed'`. Run it and count the rows:

In [ ]:
SELECT u.user_id, u.name, COALESCE(SUM(o.amount), 0) AS total
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
WHERE o.status = 'completed'
GROUP BY u.user_id, u.name;

Only 8 rows are left: the two users with no orders have disappeared.

The reason: `WHERE` runs **after** the join. For users with no orders, `o.status` is NULL, `NULL = 'completed'` is not true, and the row is filtered out. **The left join has effectively become an inner join.**

The fix is to put the condition in `ON`: filter the right table first, then join.

### ✏️ Exercise 2 (modify the query)

Fix the query above: all 10 users must appear, with `total` = 0 for users who have no completed orders. Output `user_id`, `name`, `total`.

In [ ]:
SELECT u.user_id, u.name, COALESCE(SUM(o.amount), 0) AS total
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
WHERE o.status = 'completed'
GROUP BY u.user_id, u.name;

<details>
<summary><b>Hint</b></summary>

Delete the `WHERE` line and attach the condition to `ON`: `ON u.user_id = o.user_id AND o.status = 'completed'`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT u.user_id, u.name, COALESCE(SUM(o.amount), 0) AS total
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
 AND o.status = 'completed'
GROUP BY u.user_id, u.name;
```

</details>

> **Rule of thumb:** in a left join, filters on the **left** table go in `WHERE`; filters on the **right** table go in `ON`.

## 5. Finding "records with no ..."

"Which users never placed an order?" After a left join, keep the rows where the right table's key is NULL:

In [ ]:
SELECT u.user_id, u.name
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
WHERE o.order_id IS NULL;

### ✏️ Exercise 3 (write it yourself)

Find users with **no completed orders at all**. Output `user_id`, `name`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

"Completed" is a filter on the right table, so it goes in `ON`; "no match" is `WHERE o.order_id IS NULL`. In this data every user who ordered has at least one completed order, so the result is still Gus and Ivan.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT u.user_id, u.name
FROM users u
LEFT JOIN orders o
  ON u.user_id = o.user_id
 AND o.status = 'completed'
WHERE o.order_id IS NULL;
```

</details>

### ✏️ Exercise 4 (write it yourself)

Compute the total amount of completed orders for each `country`. Output `country`, `total`. Countries with no completed orders do not need to appear.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

An inner join is enough here, so putting the filter in `WHERE` is fine.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT u.country, SUM(o.amount) AS total
FROM users u
JOIN orders o
  ON u.user_id = o.user_id
WHERE o.status = 'completed'
GROUP BY u.country;
```

</details>

## 6. Self-join

A table joined to itself. In `employees`, `manager_id` points to another employee's `emp_id`. To get both the employee's and the manager's name, use `employees` as if it were two tables, with two aliases:

In [ ]:
SELECT e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m
  ON e.manager_id = m.emp_id;

### ✏️ Exercise 5 (write it yourself)

Compute how many direct reports each manager has. Output `manager` (the manager's name) and `n_reports`. People with no reports do not need to appear.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

Join as in the example above, but with an inner join (Alice's row, which has no manager, is not needed). Then group by manager and count.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT m.name AS manager, COUNT(*) AS n_reports
FROM employees e
JOIN employees m
  ON e.manager_id = m.emp_id
GROUP BY m.emp_id, m.name;
```

</details>

**🗣 Say it out loud:** What is the difference between INNER JOIN and LEFT JOIN? When do you use a LEFT JOIN?

<details>
<summary><b>Reference answer</b></summary>

An inner join keeps only rows that match on both sides; a left join keeps every row of the left table and fills in NULL where the right table has no match. Use a left join when every unit in the left table must appear in the result, for example when computing the conversion rate over all users or recording users with no orders as 0. One caveat: put filters on the right table in ON, otherwise the left join turns into an inner join.

**Short version:** An inner join keeps only matching rows; a left join keeps every row from the left table and fills NULLs where there's no match. I use a left join when every unit on the left must appear in the result, like users with zero orders, and I put filters on the right table in the ON clause.

</details>

## Summary

- `JOIN ... ON ...`; a left join keeps every row of the left table;
- After a left join, count with `COUNT(right_table_column)` and sum with `COALESCE(SUM(...), 0)`;
- Filters on the right table go in `ON`;
- Unmatched rows: `LEFT JOIN ... WHERE right_key IS NULL`;
- Self-join: give the same table two aliases.

On to **03d**.